# Skincare data analysis

This notebook loads the `data/skincare_log.csv` file and demonstrates exploratory analysis using pandas and DuckDB. It contains example queries and simple plots to answer the README's questions and now includes a small cell to load `data/product_results.csv` for merging and visualization.

In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

# Load CSV with pandas
df = pd.read_csv('data/skincare_log.csv', parse_dates=['date'])
df


In [ ]:
# Use DuckDB for SQL-style exploration without creating a DB file
con = duckdb.connect()
con.register('skincare_df', df)
# Example: average hydration by product
print(con.execute('SELECT product, ROUND(AVG(hydration),2) AS avg_hydration, COUNT(*) AS n_records FROM skincare_df GROUP BY product ORDER BY avg_hydration DESC').df())


In [ ]:
# Compute irritation rate by product (fraction of entries with irritation >= 3)
q = '''
SELECT
  product,
  COUNT(*) AS total,
  SUM(CASE WHEN irritation >= 3 THEN 1 ELSE 0 END) AS irritation_count,
  ROUND(100.0 * SUM(CASE WHEN irritation >= 3 THEN 1 ELSE 0 END) / COUNT(*),2) AS irritation_pct
FROM skincare_df
GROUP BY product
ORDER BY irritation_pct DESC
'''
print(con.execute(q).df())


In [ ]:
# Simple visualizations
sns.set(style='whitegrid')
agg = con.execute('SELECT product, ROUND(AVG(hydration),2) AS avg_hydration, ROUND(AVG(texture),2) AS avg_texture FROM skincare_df GROUP BY product').df()
plt.figure(figsize=(8,4))
sns.barplot(data=agg, x='avg_hydration', y='product', palette='viridis')
plt.title('Average hydration by product')
plt.xlabel('Average hydration')
plt.tight_layout()
plt.show()

# Irritation percent plot
irrit = con.execute('''SELECT product, ROUND(100.0 * SUM(CASE WHEN irritation >= 3 THEN 1 ELSE 0 END) / COUNT(*),2) AS irritation_pct FROM skincare_df GROUP BY product ORDER BY irritation_pct DESC''').df()
plt.figure(figsize=(8,4))
sns.barplot(data=irrit, x='irritation_pct', y='product', palette='magma')
plt.xlabel('Percent of entries with irritation >= 3')
plt.title('Irritation rate by product')
plt.tight_layout()
plt.show()


In [ ]:
# NEW CELL: Load product_results.csv and merge with the skincare log for combined views
prod = pd.read_csv('data/product_results.csv')
# Display the product reference table
display(prod)

# Merge on product name to bring concentration/formulation into the usage log
merged = df.merge(prod[['product','concentration','formulation']], on='product', how='left')
display(merged.head())

# Quick plot: average hydration by product with formulation label in hover (if using interactive backends)
agg2 = merged.groupby(['product','formulation'], dropna=False).agg(avg_hydration=('hydration','mean'), n=('hydration','size')).reset_index()
plt.figure(figsize=(8,4))
sns.barplot(data=agg2.sort_values('avg_hydration', ascending=False), x='avg_hydration', y='product', palette='cool')
plt.xlabel('Average hydration')
plt.title('Average hydration by product (merged with product metadata)')
plt.tight_layout()
plt.show()
